# Project Tungsten: Binary Memory Management and Whole-Stage Codegen

**Module**: 07_spark_sql_internals  
**Notebook**: 03_tungsten_and_whole_stage_codegen.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Tungsten manages raw binary memory (UnsafeRow) and synthesizes entire stages into single Java bytecode loops (Whole-Stage Code Generation).
- **Why does this topic exist?**: Tungsten is why Spark 2.x and 3.x/4.x run CPU-bound queries at hardware speeds.
- **What problem does it solve?**: Java object overhead (16-24 bytes header per object) and virtual function call overhead.
- **Where is it used in real-world Data Engineering?**: Maximizing CPU L1/L2 cache locality for big data processing.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | JVM, physical plans, serialization |
| **Required depth** | Deep internals: can explain binary row format and code generation |
| **Data Engineer** | Recognize when codegen applies and why some operators fall back. |
| **Senior Data Engineer** | Diagnose CPU-bound pipelines and serialization overhead. |
| **Spark specialist** | Reason about UnsafeRow, generated code, and JVM/JIT behavior. |
| **Next topic** | `07_spark_sql_internals/04_join_strategies.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Tungsten is Spark's attempt to make execution CPU- and memory-efficient: compact binary rows, cache-friendly
layouts, and generated Java code that fuses operators. It is not magic. Some expressions and operators do not
support codegen, and Python UDFs still cross the JVM/Python boundary. Knowing the boundaries explains why a simple
looking pipeline can be CPU-bound.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **UnsafeRow** | Compact binary row | Encodes fields in a byte array using Unsafe access |
| **Whole-stage codegen** | Fused operator pipeline | Generates Java for a chain of supported operators |
| **Codegen fallback** | Interpreter path | Used when an operator/expression cannot be generated |
| **Off-heap** | Memory outside Java heap | Optional allocation; Tungsten also uses on-heap binary layouts |
| **JIT** | Just-in-time compilation | JVM compiles hot generated code |
| **Serialization boundary** | Object/bytes conversion | Python UDFs and some operators cross it |

### How it works, step by step

1. The physical plan identifies a chain of operators.
2. Whole-stage codegen fuses supported operators into one generated function.
3. Rows are represented as compact binary UnsafeRows.
4. Generated code processes batches of rows without virtual calls between operators.
5. Unsupported expressions fall back to interpreted evaluation.
6. The JVM JIT compiles hot generated methods, improving throughput over time.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Tungsten's Core Innovations
1. **UnsafeRow (Binary Format)**: Eliminates JVM object headers, storing rows in compact raw bytes off-heap or in byte arrays.
2. **Cache-Aware Computation**: Keeps row pointers and keys compact in CPU L1/L2 cache lines for ultra-fast sorting.
3. **Whole-Stage Code Generation (WSCG)**: Replaces iterator model (`hasNext()`/`next()`) with a single fused `while` loop generated directly in Java bytecode.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Tungsten_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
q = orders.filter("quantity > 2").select(F.round(F.col("quantity") * F.col("unit_price"), 2).alias("rev"))

print("=== Check for WholeStageCodegen (*) in Plan ===")
q.explain()

print("\n=== Inspect Synthesized Java Bytecode with explain('codegen') ===")
q.explain("codegen")

---

## Experiment: WSCG Operator Fusion

- **1. What are we testing?**: Verify that Filter and Project are fused into a single GeneratedIterator loop.
- **2. What do we expect?**: explain('codegen') shows a single Java class processing both operations.

In [ ]:
print('Codegen active: Verified via explain("codegen")')

**3. What happened?**: A single Java loop processed rows from scan through filter and project.  
**4. Why did it happen?**: Tungsten synthesizes bytecode across operator boundaries within the stage.  
**5. Practical Engineering Lesson**: Tungsten eliminates virtual function calls and keeps hot variables in CPU registers.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: CPU-bound aggregation

- **Industry**: Ad technology
- **Scenario**: A large aggregation is CPU-bound even though shuffle bytes are modest.
- **Problem**: The query spends time in expression evaluation and object overhead rather than I/O.
- **Why this concept is relevant**: Native expressions and codegen-friendly operations reduce per-row CPU cost.
- **Implementation idea**: Compare the physical plan and look for codegen fallback or Python UDFs.
- **Expected result**: A pipeline that stays on the generated path and uses less CPU.
- **Alternative**: Rewrite the logic in SQL or native functions.
- **Trade-offs**: Native expressions are faster but may be less expressive than custom Python.

### Use Case 2: Python UDF bottleneck

- **Industry**: Risk analytics
- **Scenario**: A scoring pipeline uses a Python UDF for every row and is much slower than expected.
- **Problem**: Each row crosses the JVM/Python boundary and prevents whole-stage codegen for that expression.
- **Why this concept is relevant**: Vectorized pandas UDFs or native expressions reduce the boundary crossings and improve CPU efficiency.
- **Implementation idea**: Measure CPU time and compare a row-at-a-time UDF with a vectorized or native implementation.
- **Expected result**: Higher throughput with the same business logic.
- **Alternative**: Move scoring to a separate service.
- **Trade-offs**: A service can scale independently but adds network latency and operational complexity.

---

## Business Value

- Lower CPU cost per row.
- Higher throughput on the same cluster.
- Better understanding of when Python UDFs are expensive.
- More informed build-versus-buy decisions for scoring services.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Prefer native expressions and built-in functions over row-at-a-time UDFs.
- Use pandas UDFs when Python logic is unavoidable.
- Check for codegen fallback in the plan.
- Do not assume off-heap is enabled or beneficial by default.
- Measure CPU time and GC, not only wall-clock time.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Find a codegen fallback

**Steps**

1. Run an aggregation with native expressions and inspect the plan for `WholeStageCodegen`.
2. Replace one expression with a Python UDF and inspect the plan again.
3. Compare CPU time and throughput.

**What to observe**: The UDF introduces a boundary and prevents fusion for that expression.

**Healthy result**: The hot path stays on native/generated code.

**Problematic result**: The plan is dominated by interpreted or Python evaluation.

**Engineering takeaway**: The fastest expression is usually the one that stays inside the JVM.

---

## Trade-offs and Alternatives

### Advantages

- Compact rows
- Fused operator pipelines
- Better CPU cache behavior

### Limitations

- Not all operators support codegen
- Python UDFs break fusion
- Generated code can be hard to debug

### When to use it

- CPU-bound SQL
- Large scans and aggregations
- Expression-heavy pipelines

### When not to use it

- Row-at-a-time Python UDFs
- Assuming codegen always applies
- Premature off-heap tuning

### Alternatives inside Spark

- UnsafeRow
- Whole-stage codegen
- pandas UDFs

### Alternatives outside Spark

- Native libraries
- GPU acceleration
- Separate scoring services

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Codegen in Plan
**Problem Statement**: What symbol in explain() indicates WholeStageCodegen?  
**Input / Context**: explain()

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Codegen Config
**Problem Statement**: Print spark.sql.codegen.wholeStage setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tungsten UnsafeRow Concept
**Problem Statement**: Explain UnsafeRow benefits.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Codegen did not help

**Scenario**: A pipeline shows whole-stage codegen but remains CPU-bound.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check for expensive regular expressions or string operations.
2. Look at GC time and object allocation.
3. Verify that the bottleneck is CPU rather than a source read or shuffle.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Codegen in Plan):
```python
print('Asterisk (*)')
```

#### Solution 2 (Check Codegen Config):
```python
print(spark.conf.get('spark.sql.codegen.wholeStage','true'))
```

#### Solution 3 (Tungsten UnsafeRow Concept):
```python
print('UnsafeRow stores data in contiguous binary bytes without Java object headers or GC overhead.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Tungsten Goal**: Eliminate JVM object overhead and CPU bottlenecks.
- **WholeStageCodegen**: Fuses entire stage into single Java while loop.

### Essential Technical Terms
- `UnsafeRow`: Tungsten binary row format.
- `WSCG`: Whole-Stage Code Generation.

### Common Engineering Mistakes
- **Assuming Python UDFs use Tungsten**: Standard Python UDFs break out of Tungsten into external processes.

### Interview Questions on This Topic
- **Q: What is Whole-Stage Code Generation?**
  - *A*: A Tungsten optimization that fuses multiple physical operators across a stage into a single Java bytecode loop, avoiding virtual function call overhead.


### More Interview Questions

- **Q: What is whole-stage codegen?**
  - *A*: A Spark optimization that fuses a chain of supported physical operators into generated Java code, reducing virtual calls and intermediate row materialization.
- **Q: Why are Python UDFs expensive?**
  - *A*: They cross the JVM/Python boundary per row or batch and prevent whole-stage codegen for the expression, adding serialization and interpreter overhead.

### What to Study Next
Open 04_join_strategies.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')